# 03 · Conversation graph per instance

Builds the **intra-instance edges** of each instance's conversation graph:

- each node is a conversation observed on the instance;
- two conversations of the same instance are connected when they share at least
  `MIN_COMMON_USERS` authors;
- the edge weight is the number of shared authors.

Authors who take part in more than `DEGREE_CAP_USERS` conversations of an instance
are ignored **when building edges** (they stay in the data). Very active users would
otherwise connect many unrelated conversations and make the graph overly dense.

Input: `<dataset>_data.csv` from notebook 02 (columns `instance_toot_found_id`,
`conversation_id`, `author_id`).
Output: a pickled DataFrame with columns `instance_toot_found_id, source, target, weight`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, triu

In [ ]:
DATASET = "pleroma"  # or "mastodon"
DATA_DIR = Path("../../data") / DATASET

MIN_COMMON_USERS = 1
DEGREE_CAP_USERS = 100

INPUT_FILE = DATA_DIR / f"{DATASET}_data.csv"
OUTPUT_FILE = DATA_DIR / f"{DATASET}_intra_edges_{MIN_COMMON_USERS}_common_user.pkl"

In [ ]:
def edges_sparse_per_instance(df, min_common, degree_cap, *, verbose=False, return_stats=False):
    """
    Conversation co-occurrence edges per instance, via a sparse
    conversation × author incidence matrix.

    Parameters
    ----------
    df : DataFrame with columns instance_toot_found_id, conversation_id, author_id
    min_common : keep pairs of conversations sharing at least this many authors
    degree_cap : ignore authors appearing in more than this many conversations of the instance
    verbose : print why an instance produced no edges
    return_stats : also return per-instance diagnostics
    """
    base = df[["instance_toot_found_id", "conversation_id", "author_id"]].dropna().drop_duplicates()

    out, stats = [], []
    for inst, g in base.groupby("instance_toot_found_id", sort=False):
        deg = g.groupby("author_id")["conversation_id"].nunique()
        g = g[g["author_id"].isin(deg[deg <= degree_cap].index)]

        inst_stat = {
            "instance_toot_found_id": inst,
            "authors_after_cap": int(g["author_id"].nunique()),
            "convs_after_cap": int(g["conversation_id"].nunique()),
            "edges_kept": 0,
        }

        if g.empty:
            if verbose:
                print(f"[skip] {inst}: empty after degree cap")
            stats.append(inst_stat)
            continue

        conv_cat = pd.Categorical(g["conversation_id"])
        auth_cat = pd.Categorical(g["author_id"])
        A = csr_matrix(
            (np.ones(len(g), dtype=np.uint8), (conv_cat.codes.astype(np.int32), auth_cat.codes.astype(np.int32))),
            shape=(len(conv_cat.categories), len(auth_cat.categories)),
        )

        # Shared authors between conversations, strict upper triangle.
        C = triu(A @ A.T, k=1)
        C.setdiag(0)
        C.eliminate_zeros()
        coo = C.tocoo()

        keep = coo.data >= min_common
        if not keep.any():
            if verbose:
                print(f"[skip] {inst}: no pair shares {min_common}+ authors")
            stats.append(inst_stat)
            continue

        edges_i = pd.DataFrame({
            "instance_toot_found_id": inst,
            "source": conv_cat.categories[coo.row[keep]].astype(object),
            "target": conv_cat.categories[coo.col[keep]].astype(object),
            "weight": coo.data[keep].astype(int),
        })
        inst_stat["edges_kept"] = len(edges_i)
        stats.append(inst_stat)
        out.append(edges_i)

    edges = (
        pd.concat(out, ignore_index=True)
        if out
        else pd.DataFrame(columns=["instance_toot_found_id", "source", "target", "weight"])
    )
    return (edges, pd.DataFrame(stats)) if return_stats else edges

In [ ]:
toots = pd.read_csv(INPUT_FILE)
edges, stats = edges_sparse_per_instance(toots, MIN_COMMON_USERS, DEGREE_CAP_USERS, return_stats=True)
edges.to_pickle(OUTPUT_FILE)

print(f"{len(edges):,} edges across {edges['instance_toot_found_id'].nunique():,} instances → {OUTPUT_FILE}")
stats.describe()